# Frozen readouts: local computation and training incentives

We keep the ViT fixed and fit spatial readers to its four score maps. Comparing 3×3 and 1×1 kernels tests whether neighborhood information changes Binding behavior. Reweighting Conjunction examples tests whether the learned decision rule follows the supervision incentive.

In [1]:
from pathlib import Path
import sys, json, hashlib, math
from collections import Counter
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score
from scipy.stats import t as student_t
from IPython.display import display
from src.medium_visual_world.learning import MediumDataset, encode_query, environment
from src.medium_visual_world.diagnostic_followup import load_frozen_model
from src.medium_visual_world.representation_probing import query_tokens
from src.medium_visual_world.render import render
from src.medium_visual_world.world import from_dict
from src.medium_visual_world.generate import read_jsonl

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = PROJECT_ROOT/"outputs/medium_world/dataset"
OUT_DIR = PROJECT_ROOT/"outputs/medium_world/experiments/architecture_protocol"
CHECKPOINT_DIR = PROJECT_ROOT/"outputs/medium_world/experiments/representation_diagnostics/five_seeds/checkpoints"
EPOCHS,BATCH_SIZE,LR = 30,128,1e-3
REFIT_ALL_READOUTS = False  # True forces a fresh fit; otherwise train only missing readouts
FEATURE_BATCH_SIZE,EVAL_BATCH_SIZE = 48,128
SEEDS = (42,43,44,45,46)
SCHEDULES=("mixed","prereq_mixed_pair_focus")
train_data=MediumDataset(DATA_DIR/"train.jsonl")
S_ROWS={family:list(read_jsonl(OUT_DIR/"selection"/(family+".jsonl"))) for family in
        ("conjunction","binding","swaps","operator","recolor")}
print(DEVICE, len(train_data), {k:len(v) for k,v in S_ROWS.items()})

cuda 10000 {'conjunction': 1000, 'binding': 1000, 'swaps': 4000, 'operator': 1500, 'recolor': 1000}


## 1. Extract the same evidence for each reader

All readers receive the four original query-conditioned maps: two attribute maps, a joint-query map and their interaction. Comparing 3×3 and 1×1 convolutions separates access to neighboring locations from pointwise processing; the wider 1×1 model also controls approximately for parameter count.

In [2]:
class SpatialReadout(nn.Module):
    def __init__(self,kernel=3,width=16):
        super().__init__()
        self.kernel=kernel
        self.width=width
        self.conv1=nn.Conv2d(4,width,kernel,padding=kernel//2)
        self.conv2=nn.Conv2d(width,width,kernel,padding=kernel//2)
        self.head=nn.Linear(2*width,2)
    def forward(self, maps):
        z=F.gelu(self.conv1(maps))
        z=F.gelu(self.conv2(z))
        return self.head(torch.cat((z.amax((2,3)),z.mean((2,3))),dim=1))

SPECS={"spatial_3x3":(3,16),"spatial_1x1":(1,16),"spatial_1x1_wide":(1,50)}
PARAMS={name:sum(p.numel() for p in SpatialReadout(*spec).parameters()) for name,spec in SPECS.items()}
display(pd.Series(PARAMS,name="trainable_parameters").to_frame())

@torch.no_grad()
def question_maps(model,tokens,queries):
    e=model.emb(queries[:,1:4])
    joint=model.query(e.flatten(1))
    scale=tokens.shape[-1]**.5
    s1=torch.einsum("bnd,bd->bn",tokens,e[:,1])/scale
    s2=torch.einsum("bnd,bd->bn",tokens,e[:,2])/scale
    sj=torch.einsum("bnd,bd->bn",tokens,joint)/scale
    r=torch.tanh(s1)*torch.tanh(s2)
    maps=torch.stack((s1,s2,sj,r),dim=1).reshape(-1,4,16,16)
    eight=torch.stack((s1.amax(1),s1.mean(1),s2.amax(1),s2.mean(1),
                       sj.amax(1),sj.mean(1),r.amax(1),r.mean(1)),dim=1)
    return maps,eight

@torch.no_grad()
def cache_maps(model,dataset,device):
    model.eval()
    tokens=[]
    for s in range(0,len(dataset.images),FEATURE_BATCH_SIZE):
        x=dataset.images[s:s+FEATURE_BATCH_SIZE].to(device=device,dtype=torch.float32).div_(255)
        p=model.patch(x).flatten(2).transpose(1,2)
        v=model.score_norm(model.pre_norm(p)+.5*model.post_norm(model.encoder(p+model.pos)))
        tokens.append(v.float().cpu())
    tokens=torch.cat(tokens)
    maps=[]
    eights=[]
    for s in range(0,len(dataset),FEATURE_BATCH_SIZE):
        ids=dataset.image_indices[s:s+FEATURE_BATCH_SIZE]
        q=dataset.queries[s:s+FEATURE_BATCH_SIZE].to(device)
        m,e=question_maps(model,tokens[ids].to(device),q)
        maps.append(m.cpu())
        eights.append(e.cpu())
    return dict(maps=torch.cat(maps),eight=torch.cat(eights),labels=dataset.labels)

class EvalDataset:
    def __init__(self,rows):
        self.rows=rows
        cache={}
        images=[]
        ids=[]
        queries=[]
        labels=[]
        for r in rows:
            key=r["scene_key"]
            if key not in cache:
                pix=np.asarray(render(from_dict(r["scene"])).convert("RGB"),dtype=np.uint8)
                cache[key]=len(images)
                images.append(pix.transpose(2,0,1).copy())
            ids.append(cache[key])
            queries.append(encode_query(query_tokens(r["task"],r["color"],r["shape"])))
            labels.append(r["label"])
        self.images=torch.from_numpy(np.stack(images))
        self.image_indices=torch.tensor(ids)
        self.queries=torch.tensor(queries)
        self.labels=torch.tensor(labels,dtype=torch.long)
    def __len__(self):
        return len(self.rows)

ALL_SELECTION_ROWS=[r for family,rows in S_ROWS.items() for r in rows]
selection_data=EvalDataset(ALL_SELECTION_ROWS)
print("Cached selection images:",len(selection_data.images),"selection questions:",len(selection_data))

,trainable_parameters
spatial_3x3,2978
spatial_1x1,418
spatial_1x1_wide,3002


Cached selection images: 4750 selection questions: 8500


## 2. Change the supervision incentive

The canonical training examples remain unchanged. We change only the relative loss contributions of bound positives, separate-object positives and the two missing-attribute negatives, keeping Conjunction's total weight fixed. A change in the recall gap then tests whether the decision rule responds to that incentive.

In [3]:
def conjunction_stratum(r):
    if r["template"]!="conjunction":return "other"
    c,s=r["question"]["args"]
    objs=r["scene"]["objects"]
    if r["label"]:return "bound" if any(o["color"]==c and o["shape"]==s for o in objs) else "separate"
    return "shape_missing" if any(o["color"]==c for o in objs) else "color_missing"

STRATA=("bound","separate","shape_missing","color_missing")
strata=np.array([conjunction_stratum(r) for r in train_data.rows])
counts={s:int((strata==s).sum()) for s in STRATA}
TARGETS={"canonical":None,
         "binding_heavy":(.40,.10,.25,.25),
         "balanced":(.25,.25,.25,.25),
         "separate_heavy":(.10,.40,.25,.25)}

def make_weights(name):
    w=np.ones(len(train_data),dtype=np.float32)
    if TARGETS[name] is not None:
        for stratum,mass in zip(STRATA,TARGETS[name]):w[strata==stratum]=2500*mass/counts[stratum]
    return torch.from_numpy(w)

weights={name:make_weights(name) for name in TARGETS}
weight_summary=[]
for name,w in weights.items():
    c=w[strata!="other"].numpy()
    overall_ess=float(c.sum()**2/(c**2).sum())
    for stratum in STRATA:
        a=w[strata==stratum].numpy()
        weight_summary.append(dict(composition=name,stratum=stratum,n=len(a),weight=float(a[0]),
                                   relative_conjunction_mass=float(a.sum()/c.sum()),
                                   global_conjunction_kish_ess=overall_ess))
display(pd.DataFrame(weight_summary))

,composition,stratum,n,weight,relative_conjunction_mass,global_conjunction_kish_ess
0,canonical,bound,759,1.000000,0.3036,2500.000000
1,canonical,separate,491,1.000000,0.1964,2500.000000
2,canonical,shape_missing,534,1.000000,0.2136,2500.000000
3,canonical,color_missing,716,1.000000,0.2864,2500.000000
4,binding_heavy,bound,759,1.317523,0.4000,2296.200684
5,binding_heavy,separate,491,0.509165,0.1000,2296.200684
6,binding_heavy,shape_missing,534,1.170412,0.2500,2296.200684
7,binding_heavy,color_missing,716,0.872905,0.2500,2296.200684
8,balanced,bound,759,0.823452,0.2500,2415.645020
9,balanced,separate,491,1.272912,0.2500,2415.645020


## 3. Fit the readouts and evaluate their behavior

Each readout uses the same supervision and fitting budget. Missing readouts are trained; saved readouts are evaluated on the current selection questions. Their predictions remain diagnostic probes of accessible evidence, not the original ViT's learned answers.

In [4]:
VARIANTS={"spatial_3x3_canonical":("spatial_3x3","canonical"),
          "spatial_1x1_canonical":("spatial_1x1","canonical"),
          "spatial_1x1_wide_canonical":("spatial_1x1_wide","canonical"),
          "spatial_3x3_binding_heavy":("spatial_3x3","binding_heavy"),
          "spatial_3x3_balanced":("spatial_3x3","balanced"),
          "spatial_3x3_separate_heavy":("spatial_3x3","separate_heavy")}

def train_diagnostic(cache,kind,composition,seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    net=SpatialReadout(*SPECS[kind]).to(DEVICE)
    optimizer=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=0)
    w=weights[composition]
    for epoch in range(EPOCHS):
        order=torch.randperm(len(cache["labels"]),generator=torch.Generator().manual_seed(seed+epoch))
        net.train()
        for idx in order.split(BATCH_SIZE):
            m=cache["maps"][idx].to(DEVICE)
            y=cache["labels"][idx].to(DEVICE)
            logits=net(m)
            loss=(F.cross_entropy(logits,y,reduction="none")*w[idx].to(DEVICE)).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    return net.eval()

@torch.no_grad()
def predict_diagnostic(net,cache):
    probs=[]
    logits=[]
    for s in range(0,len(cache["labels"]),EVAL_BATCH_SIZE):
        scores=net(cache["maps"][s:s+EVAL_BATCH_SIZE].to(DEVICE))
        probs.append(scores.softmax(-1)[:,1].cpu().numpy())
        logits.append((scores[:,1]-scores[:,0]).cpu().numpy())
    return np.concatenate(probs), np.concatenate(logits)

def evaluation_summary(rows,prob,logit):
    df=pd.DataFrame([{k:v for k,v in r.items() if k!="scene"} for r in rows])
    df["prob_yes"]=prob
    df["logit_yes"]=logit
    df["prediction"]=(df.prob_yes>=.5).astype(int)
    df["correct"]=(df.prediction==df.label).astype(int)
    conj=df[df.group=="conjunction"]
    sr=conj[conj.case=="separate_positive"].correct.mean()
    br=conj[conj.case=="bound_positive"].correct.mean()
    shape_missing=conj[conj.case=="shape_missing"].correct.mean()
    four_balanced=conj.groupby("case").correct.mean().mean()
    binding=df[df.group=="binding"]
    bind_auc=roc_auc_score(binding.label,binding.prob_yes)
    close_auc=roc_auc_score(binding.loc[binding.adjacency=="close","label"],
                            binding.loc[binding.adjacency=="close","prob_yes"])
    far_auc=roc_auc_score(binding.loc[binding.adjacency=="far","label"],
                          binding.loc[binding.adjacency=="far","prob_yes"])
    swap=df[(df.group=="swaps")&(df.task.isin(("pair","conjunction")))&(df.heldout_created==0)]
    wide=swap.pivot(index=["pair_id","task"],columns="state",values=["label","prob_yes","logit_yes","correct"])
    wide.columns=["_".join(x) for x in wide.columns]
    wide=wide.reset_index()
    bp=wide[wide.task=="pair"]
    correct_dir=np.mean((bp.prob_yes_before-bp.prob_yes_after)>0)
    both=np.mean((bp.correct_before==1)&(bp.correct_after==1))
    cj=wide[wide.task=="conjunction"]
    leak=float((cj.prob_yes_after.ge(.5)!=cj.prob_yes_before.ge(.5)).mean())
    delta_logit=float((cj.logit_yes_before-cj.logit_yes_after).mean())
    operator=df[df.group=="operator"]
    pair_correct=operator.groupby("pair_id").correct.agg(lambda x:int((x==1).all())).mean()
    return dict(conj_same_recall=br,conj_separate_recall=sr,conj_recall_gap=br-sr,
                conj_shape_missing_specificity=shape_missing,conj_four_stratum_balanced=four_balanced,
                binding_auc=bind_auc,binding_close_auc=close_auc,binding_far_auc=far_auc,
                strict_swap_correct_direction=correct_dir,strict_swap_both_correct=both,
                conjunction_leak_rate=leak,conj_directional_logit_shift=delta_logit,operator_pair_correct=pair_correct)

RUN_DIR = OUT_DIR / "frozen_dose_response"
RUN_DIR.mkdir(parents=True,exist_ok=True)
summary=[]
for schedule in SCHEDULES:
    for seed in SEEDS:
        source = CHECKPOINT_DIR/f"{schedule}_constant_seed_{seed}_step_9000.pt"
        model, _ = load_frozen_model(source,DEVICE)
        print("Frozen:",schedule,seed,flush=True)
        folder = RUN_DIR/f"{schedule}_seed{seed}"
        folder.mkdir(parents=True,exist_ok=True)
        fit_variants = {variant for variant in VARIANTS
                      if REFIT_ALL_READOUTS or not (folder/variant/"readout.pt").is_file()}
        tr=cache_maps(model,train_data,DEVICE) if fit_variants else None
        ev=cache_maps(model,selection_data,DEVICE)
        for variant,(kind,comp) in VARIANTS.items():
            if variant in fit_variants:
                net=train_diagnostic(tr,kind,comp,20261113+seed)
            else:
                state=torch.load(folder/variant/"readout.pt",map_location="cpu",weights_only=True)
                net=SpatialReadout(*SPECS[kind]).to(DEVICE)
                net.load_state_dict(state["state_dict"])
                net.eval()
            prob,logit=predict_diagnostic(net,ev)
            metrics=evaluation_summary(ALL_SELECTION_ROWS,prob,logit)
            record=dict(schedule=schedule,seed=seed,variant=variant,kind=kind,composition=comp,
                        params=PARAMS[kind],**metrics)
            summary.append(record)
            sub=folder/variant
            sub.mkdir(parents=True,exist_ok=True)
            pd.DataFrame({"example_id":[r["example_id"] for r in ALL_SELECTION_ROWS],
                          "prob_yes": prob, "logit_yes": logit}).to_csv(sub/"selection_predictions.csv",index=False)
            if variant in fit_variants:
                torch.save({"state_dict":{k:v.detach().cpu() for k,v in net.state_dict().items()},
                            "kind":kind,"composition":comp},sub/"readout.pt")
            print(" ",variant,"gap",round(metrics["conj_recall_gap"],3),"binding AUC",round(metrics["binding_auc"],3),flush=True)
            del net
        run_path = folder/"run.json"
        run=json.loads(run_path.read_text(encoding="utf-8")) if run_path.is_file() else dict(
            source_checkpoint_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
            schedule=schedule,seed=seed,training_hash=hashlib.sha256((DATA_DIR/"train.jsonl").read_bytes()).hexdigest(),
            readout_epochs=EPOCHS,reader_lr=LR,batch_size=BATCH_SIZE,
            variant_specs=VARIANTS,parameter_counts=PARAMS,environment=environment())
        run["newly_fitted_variants"]=sorted(fit_variants)
        run["evaluation_manifest_sha256"]=hashlib.sha256(
            (OUT_DIR/"selection_and_confirmation_manifest.json").read_bytes()).hexdigest()
        run["evaluation_environment"]=environment()
        run_path.write_text(json.dumps(run,indent=2),encoding="utf-8")
        del model,tr,ev
summary=pd.DataFrame(summary)
summary.to_csv(RUN_DIR/"selection_metrics_all.csv",index=False)
display(summary.groupby(["composition","kind"])[["conj_recall_gap","binding_auc","conjunction_leak_rate"]].mean())

Frozen: mixed 42
  spatial_3x3_canonical gap 0.312 binding AUC 0.901
  spatial_1x1_canonical gap -0.02 binding AUC 0.692
  spatial_1x1_wide_canonical gap -0.008 binding AUC 0.725
  spatial_3x3_binding_heavy gap 0.404 binding AUC 0.907
  spatial_3x3_balanced gap 0.28 binding AUC 0.891
  spatial_3x3_separate_heavy gap 0.164 binding AUC 0.875
Frozen: mixed 43
  spatial_3x3_canonical gap 0.244 binding AUC 0.837
  spatial_1x1_canonical gap 0.004 binding AUC 0.642
  spatial_1x1_wide_canonical gap 0.072 binding AUC 0.703
  spatial_3x3_binding_heavy gap 0.264 binding AUC 0.851
  spatial_3x3_balanced gap 0.26 binding AUC 0.832
  spatial_3x3_separate_heavy gap 0.176 binding AUC 0.82
Frozen: mixed 44
  spatial_3x3_canonical gap 0.232 binding AUC 0.797
  spatial_1x1_canonical gap 0.036 binding AUC 0.646
  spatial_1x1_wide_canonical gap 0.02 binding AUC 0.68
  spatial_3x3_binding_heavy gap 0.272 binding AUC 0.813
  spatial_3x3_balanced gap 0.248 binding AUC 0.796
  spatial_3x3_separate_heavy gap 0.

conj_recall_gap  binding_auc  \
composition    kind                                             
balanced       spatial_3x3                0.1952     0.786031   
binding_heavy  spatial_3x3                0.2552     0.800700   
canonical      spatial_1x1                0.0260     0.667802   
               spatial_1x1_wide           0.0448     0.693146   
               spatial_3x3                0.2024     0.790224   
separate_heavy spatial_3x3                0.1288     0.774794   

                                 conjunction_leak_rate  
composition    kind                                     
balanced       spatial_3x3                     0.37725  
binding_heavy  spatial_3x3                     0.43775  
canonical      spatial_1x1                     0.20800  
               spatial_1x1_wide                0.23625  
               spatial_3x3                     0.37500  
separate_heavy spatial_3x3                     0.32400

## 4. Evaluate the dose-response

The main diagnostic is the difference between Conjunction recall on bound and separate-object positives. A smaller gap suggests less dependence on same-object association. We pair results across five seeds; the two curricula from each seed are not independent replications.

In [5]:
def paired_ci(values):
    x=np.asarray(values,dtype=float)
    n=len(x)
    h=student_t.ppf(.975,n-1)*x.std(ddof=1) / np.sqrt(n)
    return dict(mean=float(x.mean()),low=float(x.mean()-h),high=float(x.mean()+h),n=n)

pivot = summary[summary.kind=="spatial_3x3"].pivot_table(index=["seed","schedule"],
    columns="composition", values="conj_recall_gap")
pooled=pivot.groupby(level="seed").mean()
ordered=all(pooled[col].mean()>=pooled[next_col].mean() for col,next_col in
            (("binding_heavy","balanced"),("balanced","separate_heavy")))
diff=pooled.binding_heavy-pooled.separate_heavy
ci=paired_ci(diff)
G1=bool(ordered and ci["mean"]>=.10 and ci["low"]>0)
print("Ordered means:",ordered," | Binding-heavy minus separate-heavy:",ci," | G1 eligible:",G1)
by_schedule={}
for schedule, w in pivot.groupby(level="schedule"):
    by_schedule[schedule]=dict(ordered=bool(w.binding_heavy.mean()>=w.balanced.mean()>=w.separate_heavy.mean()),
                               binding_heavy_minus_separate=paired_ci(w.binding_heavy-w.separate_heavy))
print("Per-schedule contrasts (reported; the gate uses seed means):",by_schedule)
print("By-schedule gaps (not independent replicates):")
display(pivot.groupby(level="schedule").mean())
mechanism=summary[summary.composition=="canonical"].pivot(index=["seed","schedule"],
   columns="kind",values=["binding_auc","conjunction_leak_rate","conj_recall_gap"])
for metric in ("binding_auc","conjunction_leak_rate","conj_recall_gap"):
    for kind in ("spatial_1x1","spatial_1x1_wide"):
        delta=(mechanism[(metric,"spatial_3x3")] - mechanism[(metric,kind)]).groupby(level="seed").mean()
        print(metric,"3x3 minus",kind,paired_ci(delta))
(RUN_DIR/"selection_decision.json").write_text(json.dumps(dict(G1=G1,ordered=ordered,
      binding_heavy_minus_separate=ci,mean_gaps=pooled.mean().to_dict(),by_schedule=by_schedule),indent=2),encoding="utf-8")
summary.groupby(["composition","kind"])[["conj_recall_gap","binding_auc","conjunction_leak_rate"]].mean().to_csv(RUN_DIR/"descriptive_means.csv")

Ordered means: True  | Binding-heavy minus separate-heavy: {'mean': 0.12639999999999998, 'low': 0.03445983240573153, 'high': 0.21834016759426844, 'n': 5}  | G1 eligible: True
Per-schedule contrasts (reported; the gate uses seed means): {'mixed': {'ordered': True, 'binding_heavy_minus_separate': {'mean': 0.12239999999999998, 'low': 0.02271982051937811, 'high': 0.22208017948062186, 'n': 5}}, 'prereq_mixed_pair_focus': {'ordered': True, 'binding_heavy_minus_separate': {'mean': 0.1304, 'low': 0.01908705591661826, 'high': 0.24171294408338173, 'n': 5}}}
By-schedule gaps (not independent replicates):


composition,balanced,binding_heavy,canonical,separate_heavy
schedule,,,,
mixed,0.1960,0.2408,0.1984,0.1184
prereq_mixed_pair_focus,0.1944,0.2696,0.2064,0.1392


binding_auc 3x3 minus spatial_1x1 {'mean': 0.12242240000000004, 'low': 0.031055915750655777, 'high': 0.2137888842493443, 'n': 5}
binding_auc 3x3 minus spatial_1x1_wide {'mean': 0.09707780000000002, 'low': 0.027927083936551897, 'high': 0.16622851606344813, 'n': 5}
conjunction_leak_rate 3x3 minus spatial_1x1 {'mean': 0.16699999999999998, 'low': 0.06021283555726886, 'high': 0.2737871644427311, 'n': 5}
conjunction_leak_rate 3x3 minus spatial_1x1_wide {'mean': 0.13875, 'low': 0.04567695051374343, 'high': 0.2318230494862566, 'n': 5}
conj_recall_gap 3x3 minus spatial_1x1 {'mean': 0.17639999999999997, 'low': 0.06275687638392675, 'high': 0.2900431236160732, 'n': 5}
conj_recall_gap 3x3 minus spatial_1x1_wide {'mean': 0.15760000000000002, 'low': 0.06403091645093714, 'high': 0.25116908354906287, 'n': 5}
